**Part 1 · AI Fundamentals** · step 1 of 25  
← start · [🏠 Roadmap](../README.md) · [02 From LLMs to agents →](02_from_llms_to_agents.ipynb)

# 01 · How Large Language Models Work

Before any framework, understand the thing every framework wraps: **the LLM API call**.

**What you'll learn**
1. What an LLM actually does (next-token prediction)
2. Tokens, context window and cost
3. Parameters: `temperature`, `max_tokens`
4. Message roles: `system`, `user`, `assistant`
5. Providers & API keys
6. Calling a model with the raw OpenAI SDK
7. Why LLMs are **stateless**, and their other limits → why we need tools, memory and agents

## 1. What is an LLM?

A **Large Language Model** is a neural network trained on huge amounts of text to do one thing:
**predict the next token** given the tokens so far. Generating an answer = predicting one token,
appending it, and repeating until a stop token.

```
"The capital of India is"  ──►  model  ──►  " New"  ──►  " Delhi"  ──►  "."  ──►  <stop>
```

- **Base model** — only continues text.
- **Chat / instruction-tuned model** (GPT-4o, Gemini, Llama-Instruct, Qwen…) — further trained to
  follow instructions and hold a conversation. These are what we use.
- **Reasoning models** (gpt-5, Qwen3 with `<think>`…) — trained to "think" step by step before answering;
  slower and pricier, better at hard problems.

## 2. Tokens, context window and cost

- A **token** is a chunk of text (roughly ¾ of an English word, ~4 characters).
- Models read and write tokens, and providers **bill per token** (input and output priced separately).
- The **context window** is the maximum number of tokens (input + output) the model can handle in one call,
  e.g. ~128k for gpt-4o. Everything the model "knows" about your conversation must fit in it.

In [1]:
import tiktoken  # OpenAI's tokenizer (installed with langchain-openai)

encoder = tiktoken.encoding_for_model("gpt-4o")
text = "LangChain makes building LLM applications easier!"
tokens = encoder.encode(text)

print(len(text), "characters ->", len(tokens), "tokens")
print([encoder.decode([t]) for t in tokens])  # see how the text was split

49 characters -> 9 tokens
['Lang', 'Chain', ' makes', ' building', ' L', 'LM', ' applications', ' easier', '!']


## 3. Key parameters

| Parameter | What it controls | Typical value |
|---|---|---|
| `temperature` | randomness. `0` = (almost) always the most likely token; higher = more creative/varied | `0` for facts/extraction, `0.7–1` for creative writing |
| `max_tokens` | upper limit on the **output** length | depends on the task |
| `top_p` | alternative to temperature (sample only from the top-p probability mass) | leave default |

## 4. Message roles

Chat models take a **list of messages**, each with a role:
| Role | Who | Purpose |
|---|---|---|
| `system` | you, the developer | behaviour, persona, rules ("You are a helpful tutor…") |
| `user` | the end user | the question / request |
| `assistant` | the model | its previous replies (you send them back for context) |
| `tool` | your code | the result of a tool the model asked to run (notebook 02) |

## 5. Providers & API keys

| Provider | Models used in this course | Key in `.env` |
|---|---|---|
| OpenAI | gpt-4o, gpt-4o-mini, gpt-4.1, gpt-5 | `OPENAI_API_KEY` |
| Google | gemini-*-flash | `GOOGLE_API_KEY` |
| Groq | open models (Qwen, Llama) on very fast hardware | `GROQ_API_KEY` |
| Tavily | web-search API used as a tool | `TAVILY_API_KEY` |

Keys live in **one** `.env` file in the `AgenticAI/` folder (copy `.env.example`).
**Never** paste keys into notebooks — they end up in git history.

In [2]:
from dotenv import load_dotenv

# Reads API keys (OPENAI_API_KEY, GROQ_API_KEY, GOOGLE_API_KEY, TAVILY_API_KEY)
# from the .env file at the AgenticAI root. load_dotenv() walks up parent
# folders until it finds one, so every notebook can share a single .env.
load_dotenv()

True

In [3]:
import os

# Check which keys are configured (prints True/False, never the key itself)
for key in ["OPENAI_API_KEY", "GROQ_API_KEY", "GOOGLE_API_KEY", "TAVILY_API_KEY"]:
    print(f"{key:<16}", bool(os.getenv(key)))

OPENAI_API_KEY   True
GROQ_API_KEY     True
GOOGLE_API_KEY   True
TAVILY_API_KEY   True


## 6. Calling a model with the raw OpenAI SDK
This is the lowest level — no LangChain. Later notebooks wrap exactly this call.

In [4]:
from openai import OpenAI

client = OpenAI()  # reads OPENAI_API_KEY from the environment

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a concise teacher."},
        {"role": "user", "content": "Explain what an LLM is in one sentence."},
    ],
    temperature=0,
    max_tokens=100,
)

print(response.choices[0].message.content)  # the answer
print(response.usage)                         # tokens used -> what you pay for

A Large Language Model (LLM) is an advanced artificial intelligence system designed to understand and generate human-like text based on vast amounts of data.
CompletionUsage(completion_tokens=29, prompt_tokens=27, total_tokens=56, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None))


### Temperature in action — same prompt, three times each

In [5]:
def ask(prompt: str, temperature: float) -> str:
    """Send a single user message and return the reply text."""
    reply = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=20,
    )
    return reply.choices[0].message.content


prompt = "Give me a name for a coffee shop. Only the name."
for t in (0, 1.2):
    print(f"temperature={t}:", [ask(prompt, t) for _ in range(3)])

temperature=0: ['Brew Haven', 'Brew Haven', 'Brew Haven']
temperature=1.2: ['Brew Haven', 'Brewed Awakening', 'Brew Haven']


## 7. LLMs are stateless
Each API call is independent. The model does **not** remember previous calls —
*you* must resend the conversation every time.

In [ ]:
print(ask("My name is Ujjawal.", 0))
print(ask("What is my name?", 0))  # separate call -> it has no idea

history = [
    {"role": "user", "content": "My name is Ujjawal."},
    {"role": "assistant", "content": "Nice to meet you, Ujjawal!"},
    {"role": "user", "content": "What is my name?"},
]
reply = client.chat.completions.create(model="gpt-4o-mini", messages=history)
print(reply.choices[0].message.content)  # works, because we sent the history

## 8. What LLMs can't do on their own → why the rest of this course exists

| Limitation | Solution | Where |
|---|---|---|
| No memory between calls | resend history / checkpointer | 08, 11 |
| Knowledge cutoff, no live data | **tools** (search, APIs, DBs) | 09, 19 |
| Can't take actions | **tools + agents** | 09–10 |
| Free text is hard to parse | **structured output** | 07, 12 |
| Hallucinations | tools / retrieval for grounding, validation, human review | 14 |
| Long chats overflow the context window | summarisation / trimming | 13 |

## Glossary
| Term | Meaning |
|---|---|
| Prompt | the input you send |
| Completion / response | what the model generates |
| Token | unit of text the model reads/writes; unit of billing |
| Context window | max tokens per call (input + output) |
| Temperature | randomness of the output |
| Hallucination | confident but wrong output |
| Embedding | a vector representing meaning of text (used for search / RAG) |

---
✅ **Done with 01 How LLMs work.** Next: [02 From LLMs to agents →](02_from_llms_to_agents.ipynb)  
[🏠 Roadmap](../README.md)